# Thí nghiệm E3: GSM8K × Qwen2.5-1.5B-Instruct

GSM8K với mô hình lớn gấp ba (1,5 tỉ tham số). Mỗi candidate là một bản sao 1,5B tham số cộng nhiễu, nên đây cũng là phép thử tải của hệ thống trên GPU 12 GB. Lần 1 (đêm 24/11) dùng giới hạn 320 token: mô hình gốc chưa viết xong khoảng một phần ba số câu, nên mức tăng phần lớn là do mô hình học viết gọn lại cho kịp ghi đáp án. Lần này giới hạn là 512 token (mô hình gốc trả lời xong khoảng 96–98% số câu), và notebook báo thêm tỉ lệ đúng **trên các câu đã trả lời xong**.

Notebook chạy theo đúng luồng của người nghiên cứu trên lab HeteroES (đăng nhập lab và thêm các máy GPU: xem
`00-chuan-bi.ipynb`):

| Bước | Ở đây |
|---|---|
| Mô hình | `Model.from_hf` ghim một commit Hugging Face; **mỗi máy tự tải từ Hugging Face** và kiểm SHA-256 từng tệp |
| Dữ liệu | `Dataset.from_rows` (train / validation / test cố định, ghim mã băm) |
| Chạy | `ESTrainer.submit(lab, coordinator, workers)`: lab xếp hàng, chia candidate cho các GPU, chấm chéo (audit) |
| Nối tiếp | mỗi đoạn sau bắt đầu từ trọng số cuối đã kiểm chứng của đoạn trước (`derive_model`) |
| Đánh giá | chọn checkpoint bằng validation; tập test chỉ chấm **một lần** ở cuối, kiểm định McNemar |

Các đoạn và lý do chọn tham số nằm ở `decisions/e3-gsm8k-qwen15.json`; mã thí nghiệm trên lab ở `state/e3-gsm8k-qwen15.json`. Chạy lại
notebook sẽ gắn vào các run đã có (không chạy lại). Đặt `CHAY_MOI = True` để chạy một run ngắn mới khi demo.

## 1. Chuẩn bị: kết nối lab và xem các GPU

In [ ]:
import json
import sys
import time
from pathlib import Path

HERE = Path.cwd()
SRC = (HERE / ".." / ".." / "src").resolve()  # runtime/notebooks/experiments -> runtime/src
for path in (str(SRC), str(HERE)):
    if path not in sys.path:
        sys.path.insert(0, path)

import runner  # noqa: E402  (sets a clean Hugging Face cache first)
import tasks  # noqa: E402
from heteroes.sdk import ESConfig, ESTrainer, reward  # noqa: E402
from heteroes.sdk_lab import Dataset, Lab, Model  # noqa: E402

NAME = 'e3-gsm8k-qwen15'
CHAY_MOI = False  # True: a short new run for a live demo (a few generations) instead of re-attaching to the recorded one
COORDINATOR = '5070ti-nhan'  # the coordinator goes to a GPU (owner rule)
EXCLUDE = set()  # names of devices to leave out of a new segment

experiment = runner.Experiment(NAME + (time.strftime("-demo-%Y%m%d-%H%M") if CHAY_MOI else ""))
lab = Lab.from_session()  # python3 scripts/lab-login.py signs in once (12 hours)


def online_workers():
    return [d.name for d in lab.devices() if d.online and d.on and not d.cpu_only and d.name not in EXCLUDE]


for d in lab.devices():
    print(f"{d.name:16} {d.gpu:28} {d.vram_gb:5.1f} GB  online={d.online} ON={d.on} bận={d.busy}")

## 2. Dữ liệu

In [ ]:
splits = tasks.gsm8k()  # train less 500 seeded rows = validation; official test = test
data = Dataset.from_rows(**splits, directory=experiment.dir / "data")
print({k: len(v) for k, v in splits.items()}, "| mã băm dữ liệu", data.sha256[:16], "…")
print(splits["test"][0]["prompt"], "\n→ đáp án", splits["test"][0]["answer"])

## 3. Mô hình

`Qwen/Qwen2.5-1.5B-Instruct` được ghim theo commit `989aa7980e4c…`. Trên lab đây là mô hình **tham chiếu**: lab không giữ bản
sao, mỗi worker tự tải từ Hugging Face rồi kiểm tra mã băm từng tệp trước khi làm việc.

In [ ]:
model = Model.from_hf('Qwen/Qwen2.5-1.5B-Instruct', revision='989aa7980e4cf806f80c7fef2b1adb7bc71aa306')
print(model.id, model.revision[:12], model.source)
for file, sha in model.manifest["files"].items():
    print(f"  {file:28} {sha[:16]}…")

## 4. Bộ chấm (scorer) và reward

`boxed-number`: lấy số trong `\boxed{...}` cuối cùng của câu trả lời. Đúng đáp án được 1 điểm; có ô nhưng sai số được 0,1 điểm (thưởng định dạng); không có ô được 0.

Reward của một candidate là trung bình điểm trên lô 32 câu train. Trên tập test, **"đúng" nghĩa là điểm 1**. Bộ chấm
là loại *khai báo* (declarative) có sẵn trong lab: lab ghim tên, phiên bản và cấu hình của nó vào công thức run, nên mọi
GPU chấm bằng cùng một đoạn mã.

In [ ]:
from heteroes.v3.scoring import score

SCORER = {"name": "boxed-number", "config": {"answerField": "answer", "formatReward": 0.1}}
TEST_SCORER = SCORER  # "đúng" = score 1: the boxed number equals the answer
make_reward = dict(scorer=SCORER)

# the scorer on a few made-up answers to the first test question (the lab runs exactly this code on every GPU)
demo = splits["test"][0]
texts = [f"... so the answer is \\boxed{{{demo['answer']}}}", "... \\boxed{7}", f"the answer is {demo['answer']}"]
full = {"kind": "declarative", "version": 1, "codeSha256": None, **SCORER}
for text, points in zip(texts, score(full, texts, [demo] * len(texts))[1]):
    print(f"{points:4}  ←  {text!r}")

### Mô hình gốc trả lời thế nào?

Ba câu test đầu, giải mã tham lam bằng bộ suy luận nghiêm ngặt `strict_b22_v1` như trong thí nghiệm (tối đa 512
token).

In [ ]:
sample = splits["test"][:3]
before = runner.score_model(model.path, sample, TEST_SCORER, 512)
for row, text, s in zip(sample, before["outputs"], before["scores"]):
    print(f"[điểm {s}]", repr(text[-300:]))

## 5. Cấu hình

32 candidate mỗi thế hệ, mỗi candidate chấm trên 32 câu train; σ và α theo từng đoạn. Validation
(500 câu) được chấm mỗi 20 thế hệ trên coordinator. **Dừng sớm**: nếu validation không vượt mức tốt nhất trước đó
trong 2 lần chấm liên tiếp, notebook hủy run và giữ điểm khôi phục đã kiểm chứng mới nhất (lý do ghi vào
`state/e3-gsm8k-qwen15.json`).

In [ ]:
def make_trainer(model, segment):
    args = ESConfig(
        model=model,
        dataset=data,
        population=32,
        batch_size=32,
        generations=3 if CHAY_MOI else segment["generations"],
        alpha=segment["alpha"],
        sigma=segment["sigma"],
        max_new_tokens=512,
        eval_dtype="float16",
        decoder="strict_b22_v1",  # same tokens on every supported NVIDIA GPU
        eval_chunk=32,
        validation_interval=20,
        backup_interval=10,  # a recovery point every 10 generations (each is a full copy of the weights)
        audit_rate=0.05,
    )
    return ESTrainer(args, code_dir=experiment.dir / "code", **make_reward)


segments = experiment.segments({'alpha': 0.0005, 'sigma': 0.001, 'generations': 100, 'reason': 'tham số của Qiu et al. 2025 (ES cho LLM): σ = 0,001, α = 5·10⁻⁴, khoảng 30 candidate'})
for i, s in enumerate(segments, 1):
    print(f"đoạn {i}: α={s['alpha']} σ={s['sigma']} {s['generations']} thế hệ — {s['reason']}")

## 6. Chạy trên lab

Mỗi đoạn là một experiment trên lab. Ô này chờ đến khi các đoạn xong (xem trực tiếp trên trang Experiments của lab);
khi chạy lại notebook, các đoạn đã có chỉ được gắn lại.

In [ ]:
runs = runner.run_segments(lab, experiment, model, make_trainer, segments, COORDINATOR, online_workers)
for (segment, run, _), i in zip(runs, range(1, len(runs) + 1)):
    print(f"đoạn {i}: {run.id} {run.status()} {run.url}")

### 6.1 Tốc độ: 1 GPU so với nhiều GPU

Cùng cấu hình chạy thêm 6 thế hệ **chỉ trên một 5070 Ti** để có mốc so sánh. Thời gian một thế hệ đọc từ ledger của
coordinator (ghi mọi candidate: máy nào chấm, lúc nào xong). Thế hệ đầu (nạp mô hình) không tính, và dùng trung vị vì
các thế hệ có chấm validation lâu hơn. Audit = một máy khác chấm lại một candidate; phải ra cùng SHA-256 của câu trả lời.

In [ ]:
import statistics

alone = runner.one_gpu_run(lab, experiment, make_trainer, runs[-1][2], segments[-1], COORDINATOR) if not CHAY_MOI else None
names = {d.worker_id: d.name for d in lab.devices()}
speed = {}
for label, run in [(f"đoạn {i}", r) for i, (_, r, _) in enumerate(runs, 1)] + ([("1 GPU", alone)] if alone else []):
    measured = runner.speed(run.id, names)
    if measured is None:
        print(label, ": ledger của coordinator không có trên máy này")
        continue
    print(f"{label}: audit khớp {measured['matched']}/{measured['audits']}, candidate mỗi máy {measured['perDevice']}")
    for machines, seconds in sorted(measured["groups"].items(), key=lambda kv: -len(kv[1])):
        print(f"   {machines:48} {len(seconds):4} thế hệ, trung vị {statistics.median(seconds):6.1f} s/thế hệ")
        speed.setdefault(machines, []).extend(seconds)
speed = {m: {"generations": len(v), "medianSeconds": round(statistics.median(v), 1)} for m, v in speed.items()}
one = next((v["medianSeconds"] for m, v in speed.items() if m == COORDINATOR), None)
most = max(speed, key=lambda m: (m.count("+"), speed[m]["generations"])) if speed else None
if one and most and most != COORDINATOR:
    print(f"→ {most}: nhanh gấp {one / speed[most]['medianSeconds']:.2f} lần một 5070 Ti")

## 7. Quá trình học

In [ ]:
import matplotlib.pyplot as plt

rewards, validation, work = runner.history(runs)
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [3, 1]})
a.plot([g for g, _ in rewards], [r for _, r in rewards], color="#9db8d8", lw=1, label="reward lô train")
a.plot([g for g, _ in validation], [r for _, r in validation], "o-", color="#c2410c", label="reward validation")
for end in [sum(s["generations"] for s, _, _ in runs[:k]) for k in range(1, len(runs))]:
    a.axvline(end - 0.5, color="#999", ls=":", lw=1)
a.set_xlabel("thế hệ")
a.set_ylabel("reward")
a.legend()
names = {d.worker_id: d.name for d in lab.devices()}
b.barh([names.get(w, w) for w in work], list(work.values()), color="#2a78d6")
b.set_xlabel("candidate đã chấm")
plt.tight_layout()
plt.show()
print("validation:", [(g, round(r, 4)) for g, r in validation])

## 8. Chọn checkpoint và chấm test (một lần)

Checkpoint được chọn là cuối đoạn có reward validation cao nhất. Tập test được chấm cho mô hình gốc, cuối đoạn đầu và
checkpoint được chọn; kết quả lưu lại nên chạy lại notebook không chấm lại.

In [ ]:
ends = []
for k, (segment, run, _) in enumerate(runs, 1):
    marks = [v for v in run.results()["validation"] if v["generation"] is not None]
    ends.append((marks[-1]["reward"], k))
best = max(ends)[1]
print("validation cuối mỗi đoạn:", ends, "→ chọn đoạn", best)

done = [(s, r, m) for s, r, m in runs]
first_dir = runner.after_segment(lab, experiment, done, 1).path
best_dir = runner.after_segment(lab, experiment, done, best).path
rows = splits["test"]
base = runner.test_report(experiment, "base", -1, model.path, rows, TEST_SCORER, 512)
first = runner.test_report(experiment, "first", 1, first_dir, rows, TEST_SCORER, 512)
chosen = runner.test_report(experiment, f"segment-{best:02d}", best, best_dir, rows, TEST_SCORER, 512)
finish = {}
for label, report in (("mô hình gốc", base), ("cuối đoạn 1", first), (f"chọn (đoạn {best})", chosen)):
    ok, n = runner.correct(report)
    f = finish[label] = runner.finished(report, 'gsm8k')
    print(f"{label:16} đúng {ok}/{n} = {ok / n:6.1%}   reward {report['checkpoints'][0]['meanReward']:.4f}   "
          f"trả lời xong {f['finished']}/{n}, đúng trên số đã xong {f['rightAmongFinished'] / max(f['finished'], 1):6.1%}")

Cột cuối là phép kiểm tra của lần 1: nếu tỉ lệ đúng chỉ tăng vì mô hình **trả lời xong** nhiều câu hơn (còn tỉ
lệ đúng trên các câu đã xong thì không đổi), thì ES chỉ dạy mô hình viết gọn, không phải suy luận tốt hơn.

In [ ]:
from heteroes.v3.comparison import compare

result = compare(base, chosen)
print(f"gốc → chọn: sửa đúng {result['fixed']}, làm sai {result['broke']}, McNemar p = {result['mcnemarExactP']:.2g}, "
      f"khoảng tin cậy 95% của mức tăng reward [{result['bootstrapCI'][0]:.3f}; {result['bootstrapCI'][1]:.3f}]")
if best != 1:
    later = compare(first, chosen)
    print(f"đoạn 1 → chọn: sửa {later['fixed']}, làm sai {later['broke']}, p = {later['mcnemarExactP']:.2g}")

## 9. Ví dụ trước / sau

In [ ]:
shown = 0
for i, row in enumerate(rows):
    a, b = base["checkpoints"][0], chosen["checkpoints"][0]
    if a["scores"][i] != 1 and b["scores"][i] == 1:
        print("—", row["prompt"][:200].replace("\n", " "))
        print("  gốc :", repr(a["outputs"][i][-200:]))
        print("  sau :", repr(b["outputs"][i][-200:]))
        shown += 1
    if shown == 3:
        break

## 10. Lưu tóm tắt (cho notebook tổng hợp)

In [ ]:
summary = {
    "name": NAME, "model": model.id, "revision": model.revision, "segments": [{**s, "experimentId": r.id} for s, r, _ in runs],
    "validation": validation, "workPerDevice": {names.get(w, w): c for w, c in work.items()},
    "test": {label: dict(zip(("correct", "count"), runner.correct(rep))) | {"meanReward": rep["checkpoints"][0]["meanReward"]}
             for label, rep in (("base", base), ("first", first), ("selected", chosen))},
    "selectedSegment": best, "comparison": result, "finished": finish, "speed": speed,
    "oneGpuRun": alone.id if alone else None,
}
out = runner.HERE / "results" / f"{experiment.name}.json"
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(summary, indent=1, ensure_ascii=False) + "\n")
print("đã lưu", out)